In [ ]:
#@title 1. Benchmark / Бенчмарк { display-mode: "form" }
#@markdown Runs the pre-registered protocol against data whose truth is known. None of your data is
#@markdown involved: every observation is generated from the seed, so nothing is uploaded anywhere.
PROFILE = "quick" #@param ["quick", "standard", "full"]
SEED = 20260904 #@param {type:"integer"}
THREADS = 0 #@param {type:"integer"}
LANGUAGE = "en" #@param ["en", "ru"]
BRANCH = "main" #@param {type:"string"}

import os, subprocess, sys, time, glob, zipfile, shutil, textwrap, shlex

REPO_URL = "https://github.com/d1d2dopamine/MVS-Analyzer.git"
ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
SRC = os.path.join(ROOT, "MVS-Analyzer")
BIN = os.path.join(ROOT, "mvs-bin")
OUT = os.path.join(ROOT, "mvs-out")
DOTNET_DIR = os.path.join(ROOT, "dotnet")
DOTNET = os.path.join(DOTNET_DIR, "dotnet")
MVS = os.path.join(BIN, "mvs")

def q(value):
    return shlex.quote(str(value))

def run(command, cwd=None, quiet=False):
    if not quiet:
        print("$ " + command)
    finished = subprocess.run(command, shell=True, cwd=cwd, text=True,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if finished.stdout:
        print(finished.stdout.rstrip())
    if finished.returncode != 0:
        raise SystemExit("The step above failed with exit code %d." % finished.returncode)
    return finished.stdout

def step(number, text):
    print("")
    print("[%s] %s" % (number, text))

start = time.time()

step(1, "Fetching the source")
if not os.path.isdir(SRC):
    run("git clone --depth 1 --branch %s %s %s" % (q(BRANCH), q(REPO_URL), q(SRC)))
else:
    print("already present: " + SRC)

step(2, "Installing the .NET 8 build tools")
os.environ["DOTNET_CLI_TELEMETRY_OPTOUT"] = "1"
os.environ["DOTNET_NOLOGO"] = "1"
os.environ["DOTNET_SKIP_FIRST_TIME_EXPERIENCE"] = "1"
if not os.path.exists(DOTNET):
    print("this takes a minute or two, and only on the first run of a session")
    run("curl -sSL https://dot.net/v1/dotnet-install.sh -o /tmp/dotnet-install.sh")
    run("bash /tmp/dotnet-install.sh --channel 8.0 --install-dir %s --no-path" % q(DOTNET_DIR))
else:
    print("already present: " + DOTNET)

step(3, "Building the headless analyzer")
if not os.path.exists(MVS):
    run("%s publish %s -c Release -o %s --nologo -v minimal"
        % (q(DOTNET), q(os.path.join(SRC, "MvsAnalyzer.Cli", "MvsAnalyzer.Cli.csproj")), q(BIN)))
else:
    print("already built: " + MVS)
run("chmod +x " + q(MVS), quiet=True)
run("%s version" % q(MVS))
run("%s env" % q(MVS))

step(4, "Running the protocol")
print("A free session gives out two cores. Expect quick to take longer here than on a laptop with")
print("more of them; what a session buys is twelve uninterrupted hours and a machine you can leave.")
print("standard is a few hours. full is longer than a free session lasts.")
BENCHMARK = os.path.join(OUT, "benchmark")
os.makedirs(BENCHMARK, exist_ok=True)

command = ("%s benchmark --profile %s --seed %d --out %s --lang %s"
           % (q(MVS), q(PROFILE), int(SEED), q(BENCHMARK), q(LANGUAGE)))
if int(THREADS) > 0:
    command += " --threads %d" % int(THREADS)

print("$ " + command)
finished = subprocess.run(command, shell=True, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(finished.stdout.rstrip())
EXIT_CODE = finished.returncode
print("")
if EXIT_CODE == 0:
    print("Every threshold was met.")
elif EXIT_CODE == 2:
    print("The run completed and at least one threshold was missed. That is a result, not a crash.")
else:
    raise SystemExit("The benchmark failed with exit code %d." % EXIT_CODE)
print("Finished in %d seconds. Now run the second cell." % int(time.time() - start))

In [ ]:
#@title 2. Read the verdicts / Вердикты { display-mode: "form" }
#@markdown The hypotheses the protocol was written to test, and what this run said about them.
folders = sorted(glob.glob(os.path.join(BENCHMARK, "MVS_Benchmark_*")))
if not folders:
    raise SystemExit("No benchmark folder was found. Run the first cell.")
FOLDER = folders[-1]
print("folder  " + FOLDER)
print("")

for name in ["benchmark_verdicts.csv", "benchmark_summary.csv"]:
    path = os.path.join(FOLDER, name)
    if not os.path.exists(path):
        continue
    try:
        import pandas
        print(name)
        display(pandas.read_csv(path))
        print("")
    except Exception as error:
        print("Could not render %s (%s)." % (name, error))

manifest = os.path.join(FOLDER, "benchmark_manifest.json")
if os.path.exists(manifest):
    import json as _json
    recorded = _json.load(open(manifest))
    for key in ["overall", "protocolHash", "protocolUnchanged", "environmentHash",
                "determinismScope", "threads", "durationSeconds"]:
        if key in recorded:
            print("%-20s %s" % (key, recorded[key]))

report = os.path.join(FOLDER, "benchmark_report.md")
if os.path.exists(report):
    print("")
    print("---- benchmark_report.md ----")
    print(open(report, encoding="utf-8").read())

In [ ]:
#@title 3. Download the results / Скачать результаты { display-mode: "form" }
#@markdown Packs the calibration, the analysis, the tables and the manifests into one archive.
import hashlib, datetime

stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M%S")
made = shutil.make_archive(os.path.join(ROOT, "MVS_results_" + stamp), "zip", OUT)

print("archive   " + made)
print("size      %.1f KB" % (os.path.getsize(made) / 1024.0))
print("sha256    " + hashlib.sha256(open(made, "rb").read()).hexdigest())
print("")
print("The manifest inside records the formula hash, the seed and the environment id, so this")
print("archive can be checked against a run made on your own machine.")

try:
    from google.colab import files
    files.download(made)
except Exception as error:
    print("")
    print("Automatic download did not start (%s)." % error)
    print("Open the folder icon in the left sidebar and download the archive by hand.")